In [4]:
from pymilvus import MilvusClient
import numpy as np
from tqdm import tqdm
from time import time
import pandas as pd
from typing import List, Dict

from scipy.spatial.distance import cosine
from sentence_transformers import SentenceTransformer

In [6]:
client = MilvusClient("../../rag_v1_milvus.db")
collection_name = "rag_v1"

2025-05-19 11:11:28,362 [ERROR][_create_connection]: Failed to create new connection using: 5cae7571ed9f403e90b35abf15eb70c3 (milvus_client.py:923)


MilvusException: <MilvusException: (code=2, message=Fail connecting to server on unix:/tmp/tmpv34wxpuw_rag_v1_milvus.db.sock, illegal connection params or server unavailable)>

In [3]:
model = SentenceTransformer("intfloat/multilingual-e5-large-instruct")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [4]:
def compute_mmr_score(query_vec, docs_vecs, lambda_param=0.5):
    selected = [0]  # on suppose que le meilleur doc est le premier
    mmr_scores = []
    for i in range(1, len(docs_vecs)):
        sim_to_query = 1 - cosine(query_vec, docs_vecs[i])
        sim_to_selected = max([1 - cosine(docs_vecs[i], docs_vecs[j]) for j in selected])
        mmr = lambda_param * sim_to_query - (1 - lambda_param) * sim_to_selected
        mmr_scores.append(mmr)
    return np.mean(mmr_scores) if mmr_scores else 0.0

## Dense Retrieval

In [26]:
def dense_retrieval(query: str, top_k: int = 10) -> dict:
    start = time()

    # Vectorisation de la requête
    query_vec = model.encode(query, normalize_embeddings=True).tolist()

    # Recherche dans Milvus
    results = client.search(
        collection_name="rag_v1",
        data=[query_vec],
        anns_field="vector",
        search_params={"metric_type": "COSINE"},
        limit=10,
        output_fields=["text", "vector"]
    )

    end = time()
    query_time = round(end - start, 4)

    # Récupération des résultats
    hits = results[0]
    if not hits:
        return {
            "method": "dense",
            "cosine@1": 0.0,
            "mmr_score": 0.0,
            "query_time": query_time,
            "ragas_like_score": 0.0,
        }

    # Vecteurs des documents
    docs_vecs = [hit["entity"]["vector"] for hit in hits]
    top1_vec = docs_vecs[0]
    cosine_sim_1 = round(1 - cosine(query_vec, top1_vec), 4)

    # Score MMR
    mmr_score = round(compute_mmr_score(query_vec, docs_vecs), 4)

    # Score agrégé (Ragas-like)
    ragas_like = round(0.5 * cosine_sim_1 + 0.3 * mmr_score + 0.2 * (1 - query_time), 4)

    return {
        "method": "dense",
        "cosine@1": cosine_sim_1,
        "mmr_score": mmr_score,
        "query_time": query_time,
        "ragas_like_score": ragas_like,
    }

In [28]:
result_dense = dense_retrieval("Quel est l'impact du changement climatique sur la biodiversité ?", top_k=10)
print(result_dense)

{'method': 'dense', 'cosine@1': 0.8341, 'mmr_score': -0.0022, 'query_time': 0.2238, 'ragas_like_score': 0.5716}


## Multi Query Fusion Retrieval

In [35]:
def multi_query_fusion_lite(query: str, variations: List[str], top_k: int = 10) -> Dict:
    start = time()

    all_queries = [query] + variations
    all_vectors = model.encode(all_queries, normalize_embeddings=True)

    result_scores = {}
    doc_embeddings = {}
    doc_texts = {}

    for vec in all_vectors:
        results = client.search(
            collection_name="rag_v1",
            data=[vec.tolist()],
            anns_field="vector",
                search_params={"metric_type": "COSINE"},
            limit=top_k,
            output_fields=["text", "vector"]
        )[0]

        for hit in results:
            doc_id = hit["id"]
            score = 1 - cosine(vec, hit["entity"]["vector"])
            result_scores[doc_id] = result_scores.get(doc_id, 0) + score
            doc_embeddings[doc_id] = hit["entity"]["vector"]
            doc_texts[doc_id] = hit["entity"]["text"]

    for doc_id in result_scores:
        result_scores[doc_id] /= len(all_queries)

    sorted_docs = sorted(result_scores.items(), key=lambda x: -x[1])
    top_docs = sorted_docs[:top_k]

    query_vec = all_vectors[0]
    top_vectors = [doc_embeddings[doc_id] for doc_id, _ in top_docs]

    # CosineSim@1
    cosine_sim_1 = round(1 - cosine(query_vec, top_vectors[0]), 4)

    # MMR
    mmr_score = round(compute_mmr_score(query_vec, top_vectors), 4)

    # Temps total
    query_time = round(time() - start, 4)

    # Score global
    ragas_like = round(0.5 * cosine_sim_1 + 0.3 * mmr_score + 0.2 * (1 - query_time), 4)

    return {
        "method": "fusion",
        "cosine@1": cosine_sim_1,
        "mmr_score": mmr_score,
        "query_time": query_time,
        "ragas_like_score": ragas_like,
    }

In [31]:
variations = [
    "Impact du réchauffement climatique sur les écosystèmes",
    "Conséquences du climat sur la biodiversité",
    "Comment le climat change-t-il la vie naturelle ?"
]

In [36]:
result_mq_fusion = multi_query_fusion_lite(
    query="Quel est l'impact du changement climatique sur la biodiversité ?",
    variations=variations,
    top_k=10
)
print(result_mq_fusion)

{'method': 'fusion', 'cosine@1': 0.8193, 'mmr_score': -0.0223, 'query_time': 0.4213, 'ragas_like_score': 0.5187}


## Multi Query Fusion Retrieval avec MMR

In [38]:
def multi_query_mmr_lite(query: str, variations: List[str], top_k: int = 10, lambda_param=0.5) -> Dict:
    start = time()

    # Vectoriser la requête principale + ses variations
    all_queries = [query] + variations
    all_vectors = model.encode(all_queries, normalize_embeddings=True)

    query_vec = all_vectors[0]
    all_hits = {}

    # Collecter tous les documents retournés par les variations
    for vec in all_vectors:
        results = client.search(
            collection_name="rag_v1",
            data=[vec.tolist()],
            anns_field="vector",
            search_params={"metric_type": "COSINE"},
            limit=top_k,
            output_fields=["text", "vector"]
        )[0]

        for hit in results:
            doc_id = hit["id"]
            if doc_id not in all_hits:
                all_hits[doc_id] = {
                    "vector": hit["entity"]["vector"],
                    "text": hit["entity"]["text"],
                    "score": 1 - hit["distance"]
                }
            else:
                all_hits[doc_id]["score"] += 1 - hit["distance"]  # accumulate score

    # Moyenne des scores pour chaque doc
    for doc in all_hits.values():
        doc["score"] /= len(all_queries)

    # Appliquer MMR pour reranker les documents fusionnés
    doc_items = list(all_hits.items())
    docs_vecs = [item[1]["vector"] for item in doc_items]
    scores = [item[1]["score"] for item in doc_items]
    doc_ids = [item[0] for item in doc_items]

    selected = []
    while len(selected) < min(top_k, len(docs_vecs)):
        best_score = -float('inf')
        best_idx = -1
        for i, doc_vec in enumerate(docs_vecs):
            if i in selected:
                continue
            sim_to_query = 1 - cosine(query_vec, doc_vec)
            sim_to_selected = max([1 - cosine(doc_vec, docs_vecs[j]) for j in selected], default=0)
            mmr_score = lambda_param * sim_to_query - (1 - lambda_param) * sim_to_selected
            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = i
        selected.append(best_idx)

    top_selected_vecs = [docs_vecs[i] for i in selected]
    cosine_sim_1 = round(1 - cosine(query_vec, top_selected_vecs[0]), 4)
    mmr_score = round(compute_mmr_score(query_vec, top_selected_vecs), 4)
    query_time = round(time() - start, 4)
    ragas_like = round(0.5 * cosine_sim_1 + 0.3 * mmr_score + 0.2 * (1 - query_time), 4)

    return {
        "method": "mmr",
        "cosine@1": cosine_sim_1,
        "mmr_score": mmr_score,
        "query_time": query_time,
        "ragas_like_score": ragas_like,
    }

# Exécution avec les mêmes variations
result_mmr = multi_query_mmr_lite(
    "Quel est l'impact du changement climatique sur la biodiversité ?",
    variations,
    top_k=10
)

In [40]:
result_mq_fusion_mmr = multi_query_mmr_lite(
    query="Quel est l'impact du changement climatique sur la biodiversité ?",
    variations=variations,
    top_k=10
)
print(result_mq_fusion_mmr)

{'method': 'mmr', 'cosine@1': 0.8556, 'mmr_score': -0.0128, 'query_time': 0.9, 'ragas_like_score': 0.444}


## Comparaison des méthodes

In [41]:
def benchmark_query_all(query: str, variations: List[str], top_k: int = 10) -> pd.DataFrame:
    results = []

    # Méthode 1 : Dense Retrieval
    try:
        dense_result = dense_retrieval(query, top_k=top_k)
        results.append(dense_result)
    except Exception as e:
        results.append({"method": "dense", "error": str(e)})

    # Méthode 2 : Multi-query fusion pondérée
    try:
        fusion_result = multi_query_fusion_lite(query, variations, top_k=top_k)
        results.append(fusion_result)
    except Exception as e:
        results.append({"method": "fusion", "error": str(e)})

    # Méthode 3 : Multi-query fusion avec MMR
    try:
        mmr_result = multi_query_mmr_lite(query, variations, top_k=top_k)
        results.append(mmr_result)
    except Exception as e:
        results.append({"method": "mmr", "error": str(e)})

    return pd.DataFrame(results)

In [42]:
benchmark_df = benchmark_query_all(
    "Quel est l'impact du changement climatique sur la biodiversité ?",
    variations,
    top_k=10
)
benchmark_df

,method,cosine@1,mmr_score,query_time,ragas_like_score
0,dense,0.8341,-0.0022,0.5001,0.5164
1,fusion,0.8193,-0.0223,0.9411,0.4147
2,mmr,0.8556,-0.0128,0.7578,0.4724


In [43]:
benchmark_df = benchmark_query_all(
    "Quel est l'impact du changement climatique sur la biodiversité ?",
    variations,
    top_k=100
)
benchmark_df

,method,cosine@1,mmr_score,query_time,ragas_like_score
0,dense,0.8341,-0.0022,0.2488,0.5666
1,fusion,0.8193,-0.0297,1.6509,0.2706
2,mmr,0.8556,-0.0216,85.5457,-16.4878


In [44]:
benchmark_df = benchmark_query_all(
    "Quel est l'impact du changement climatique sur la biodiversité ?",
    variations,
    top_k=1
)
benchmark_df

,method,cosine@1,mmr_score,query_time,ragas_like_score
0,dense,0.8341,-0.0022,0.2723,0.5619
1,fusion,0.8556,0.0000,0.4279,0.5422
2,mmr,0.8556,0.0000,0.4282,0.5422
